# Lekcija 18: Osiguravanje AI agenata s kriptografskim potvrđivanjem

## Praktični bilježnik

Ovaj bilježnik prolazi kroz četiri vježbe:

1. **Potpišite svoj prvi račun** za poziv alata agenta i provjerite ga.
2. **Manipulirajte računom** i promatrajte neuspjeh provjere.
3. **Izgradite lanac od tri računa** i potvrdite integritet lanca.
4. **Omotajte poziv alata Microsoft Agent Frameworka** tako da svaki korak emitira račun.

Sve kriptografske primitivne funkcije uvoze se iz dobro održavanih biblioteka (`pynacl` za Ed25519, `jcs` za RFC 8785 kanonski JSON, `hashlib` iz standardne Python biblioteke za SHA-256). Sama logika računa je običan Python koji možete čitati i mijenjati.

Pokrenite ćelije redom. Svaki dio je kratak i samostalan.


## Postavljanje

Instalirajte dvije ovisnosti. Obje imaju permisivne licence (Apache-2.0 / MIT).


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## Pomoćni alati

Ova dva pomoćna alata obrađuju base64url kodiranje (bez popunjavanja) i SHA-256 heširanje proizvoljnih objekata. Ostatak bilježnice ostavljaju usredotočen na samu logiku primitka.


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Section 1: Potpišite svoj prvi račun

Zamislite da je naš agent za **Contoso Travel** upravo potražio letove od Sydneyja do Los Angelesa za kupca. Želimo zabilježiti ovaj poziv alatu kao potpisani račun kako bi budući revizor mogao to provjeriti bez potrebe da nam vjeruje.

### Step 1.1: Generirajte ključ za potpisivanje

U produkciji, ključ za potpisivanje agenta bi se nalazio u hardverskom sigurnosnom modulu (HSM), Azure Key Vaultu ili sličnom zaštićenom spremištu. Za ovu lekciju generiramo svježi ključ u memoriji.


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### Korak 1.2: Izgradnja sadržaja potvrde

Sadržaj sadrži sve što želimo da potvrda potvrdi: tko je djelovao, koji je alat korišten, s kojim argumentima, što je rezultat, pod kojom politikom i kada. Heširamo argumente i rezultat umjesto da ih unosimo izravno kako potvrda ne bi otkrila osjetljiv sadržaj.


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### Korak 1.3: Potpišite i sastavite priznanicu

Tri koraka:

1. Kanonizirajte teret pomoću JCS-a tako da dvije implementacije koje proizvode istu logičku priznanicu proizvode bajtove identične na razini bajtova.
2. Sažmite kanonizirane bajtove s SHA-256.
3. Potpišite sažetak privatnim ključem Ed25519.

Potpis se zatim pridružuje izvornom teretu kako bi se proizvela konačna priznanica.


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### Korak 1.4: Potvrdite primitak

Verifikacija je obrnut proces. Uklonimo potpis, ponovno izračunamo kanonski hash i provjeravamo potpis u odnosu na javni ključ u primitku.

Revizor koji vrši ovu provjeru ne treba ništa od nas osim sam primitak. Nema potrebe za pozivanjem usluge, upitom direktorija ključeva ili povjerenjem.


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

Trebali biste vidjeti `Receipt is valid: True`. Agent je proizveo svoj prvi kriptografski potpisani zapis revizije.


## Odjeljak 2: Manipulacija računom

Cijela svrha računa je da budu otporni na manipulaciju. Dokazat ćemo to.

Modificirat ćemo točno jedan znak na računu i promatrati kako verifikacija ne uspije.


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### Što se upravo dogodilo?

Kad smo promijenili `policy_id`, kanonski bajtovi su se promijenili. SHA-256 hash tih bajtova se promijenio. Potpis (koji je bio nad izvornim hashom) više ne odgovara novom hashu. Verifikacija ispravno vraća `False`.

Nema načina za izmjenu bilo kojeg polja računa i da se i dalje potvrđuje, osim ako napadač nema privatni ključ. Dok god je privatni ključ u skladištu ključeva, a javni ključ je objavljen, nemoguće je sakriti mijenjanje.

Isprobajte sami: promijenite `tool_name` ili `agent_id` ili `timestamp` u gornjoj ćeliji i ponovno pokrenite. Svaka promjena proizvodi nevažeći račun.


## Section 3: Povežite račune u lanac

Jedan račun štiti jednu radnju. Većina agenata izvršava mnogo radnji. Kako bismo cijeli slijed učinili otkrivajućim manipulaciju, povezujemo svaki račun s prethodnim tako da u nositelj novog računa uključimo hash prethodnog računa.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Ako netko ukloni ili preuredi račun, lanac se prekida upravo na toj točki. Verifikacija bilo kojeg kasnijeg računa ne uspijeva jer njegov `previous_receipt_hash` više ne odgovara stvarnom hashu njegovog prethodnika.


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Sad prekini lanac tako da promijeniš srednji račun i ponovno provjeriš. Promijenjeni račun ne prođe provjeru potpisa, A sljedeći račun ne prođe provjeru veze u lancu (jer njegov `previous_receipt_hash` više ne odgovara izmijenjenom hash-u srednjeg računa).


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Primka 0 se i dalje provjerava (nije modificirana i nema prethodnika na koji bi se oslanjala). Primka 1 ne prolazi provjeru potpisa jer smo promijenili `tool_args_hash`. Primka 2 ne prolazi provjeru lanca veza jer je njezin `previous_receipt_hash` izračunat na temelju originalne (sada modificirane) primke 1.

Čak i ako napadač ponovno potpiše modificiranu primku 1 (što ne može učiniti bez privatnog ključa), neslaganje veze lanca u primci 2 i dalje bi otkrilo manipulaciju. Da bi prikrio promjenu, napadač bi morao ponovno potpisati svaku primku od točke izmjene nadalje, što zahtijeva posjedovanje privatnog ključa.


## Sekcija 4: Omotavanje poziva agentnog alata s potpisivanjem primitka

U stvarnoj implementaciji ne želite da svaki autor agenta mora pamtiti pozvati `make_receipt`. Želite da potpisivanje primitka bude automatsko za svaki poziv alata.

Ovo je najjednostavniji obrazac: omotavajuća klasa koja prima bilo koju pozivu alatnu funkciju i vraća verziju koja emitira primitak. Ovo se prilagođava bilo kojem agentskom okviru, uključujući Microsoft Agent Framework (`agent_framework.azure`).

Ako nemate postavljen Azure AI Foundry projekt, lokalni primjer u nastavku i dalje prikazuje obrazac.


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to AzureAIProjectAgentProvider as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")

### Integracija s Microsoft Agent Frameworkom

Gornji omotač `ReceiptedTool` ne ovisi o frameworku. Za korištenje unutar agenta izgrađenog s Microsoft Agent Frameworkom, registrirajte omotanu funkciju kao alat. Skica (zamijenili biste lažni prikaz stvarnom registracijom alata Azure AI Foundry):

```python
# Pseudokod koji prikazuje oblik integracije.
# from agent_framework.azure import AzureAIProjectAgentProvider
# from azure.identity import AzureCliCredential
#
# provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())
# agent = provider.create_agent(
#     instructions="Vi ste agent Contoso Travel ...",
#     tools=[receipted_lookup],   # omotani alat, a ne sirova funkcija
# )
# response = agent.run("Pronađi letove od Sydneya do Los Angelesa u lipnju.")
#
# # Nakon izvođenja, svaki poziv alata koji je agent napravio ima potpisani račun:
# audit_chain = receipted_lookup.receipts
```

Agent framework ne treba znati ništa o potvrda. Potpisivanje potvrda je omotano oko alata, a ne ugrađeno u framework. Na ovaj način dodajete izvornost postojećem kodu agenta bez prepisivanja agenta.


## Sažetak i izazov za daljnje vježbanje

Imate:

- Generirani Ed25519 par ključeva.
- Sastavljen i potpisan potvrdu za poziv agentnog alata.
- Potvrdu verificiranu offline koristeći samo javni ključ.
- Manipuliranu potvrdu i uočeni neuspjeh u provjeri.
- Sastavljen niz potvrda povezanih hashovima u lancu od tri potvrde.
- Manipulaciju sredine lanca i uočene neuspjehe potpisa i veze u lancu.
- Omatanje funkcije alata s automatskim potpisivanjem potvrda.

**Izazov za daljnje vježbanje.** Proširite shemu potvrde poljem `request_id` (UUID za distribuirano praćenje). Ažurirajte `make_receipt` da ga uključi, i potvrdite da potvrde i dalje prolaze provjeru od početka do kraja. Zatim izmijenite to polje nakon potpisivanja i potvrdite da provjera ne uspijeva. Ovo vas prisiljava da internalizirate kako svaki bajt kanonskog kodiranja doprinosi potpisu.

**Važna granica.** Potvrde dokazuju tri stvari i samo tri stvari: pripadnost (ovaj ključ je potpisao ovaj sadržaj), integritet (sadržaj se nije promijenio od potpisivanja) i redoslijed (ova potvrda je nastala nakon one potvrde). One NE dokazuju da je akcija agenta bila ispravna, da je pravilo nazvano u `policy_id` doista evaluirano, ili da je agent slijedio svako pravilo. Potvrde su temelj. Upravljanje je sustav koji gradite na tome.

Ponovno pročitajte README lekcije imajući tu granicu na umu. Najčešća pogreška timova s potvrđama jest pretpostavka da "imamo potvrde" znači "mi smo upravljani". To nije tako. Potvrde čine ponašanje agenta auditabilnim. One ga ne čine ispravnim.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Napomena**:
Ovaj dokument je preveden korištenjem AI prevoditeljskog servisa [Co-op Translator](https://github.com/Azure/co-op-translator). Iako težimo točnosti, imajte na umu da automatski prijevodi mogu sadržavati greške ili netočnosti. Izvorni dokument na izvornom jeziku treba smatrati autoritativnim izvorom. Za važne informacije preporuča se profesionalni ljudski prijevod. Nismo odgovorni za bilo kakva nesporazumevanja ili pogrešne interpretacije koje proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
